<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!git clone https://github.com/yaranoun/ML-Tech.git

fatal: destination path 'ML-Tech' already exists and is not an empty directory.


In [41]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 21 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (21/21), 6.68 KiB | 621.00 KiB/s, done.
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
   de27d4c..22ee941  main       -> origin/main
Updating de27d4c..22ee941
Fast-forward
 data/processed/chunks.json                         |   8 +-
 ...30\250\331\206\330\247\331\206\331\212\330\251" |   6 +-
 ...43\330\254\331\206\330\250\331\212\330\251.txt" |   4 +-
 notebooks/02_embeddings.ipynb                      | 320 +++++++++------------
 4 files changed, 140 insertions(+), 198 deletions(-)


In [42]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 33 chunks
{'document': 'استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي.txt', 'title': '"استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي(يستوجب حضور صاحب العلاقة شخصيا)(للاطلاع على المستندات المطلوبة دون الحاجة لحجز موعد مسبق)"', 'url': 'https://tmo.gov.lb/web/panel/info/service-types/1', 'category': "Driver's Licence", 'service': 'رخصة سوق عمومية للضمان الاجتماعي', 'language': 'ar', 'keywords': "Driver's license, driving license, Lebanon, Lebanese, documents, appointment,taxi,cab, public, social security,رخصة سوق، لبنان، لبناني، المستندات ، عمومي، ضمان اجتماعي، موعد،", 'section': 'الوصف', 'text': 'لا يتطلب موعد\n(يستوجب حضور صاحب العلاقة شخصيا)\n1.1\nاستمارة تُطلب من قبل الصندوق الوطني للضمان الاجتماعي للمستفيدين من خدماته من فئة السائقين العموميين، وذلك للتأكّد من تجديدهم لرخصة السوق بصورة منتظمة.\n\n2'}


In [43]:
!pip install -q sentence-transformers faiss-cpu

In [44]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [45]:
passages = [
    "passage: "
    + "Service: " + chunk.get("service", "") + "\n"
    + "Service: " + chunk.get("service", "") + "\n"
    + "Section: " + chunk.get("section", "") + "\n"
    + "Section: " + chunk.get("section", "") + "\n"
    + "Keywords: " + chunk.get("keywords", "") + "\n"
    + "Content: " + chunk["text"]
    for chunk in chunks
]

In [46]:
embeddings = embedding_model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(33, 768)


In [47]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [48]:
faiss.write_index(
    index,
    "data/processed/passport_index.faiss"
)

In [49]:
import os

file_path = "data/processed/passport_index.faiss"
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"Error: The file '{file_path}' does not exist. Please re-run the cell that creates it (cell `CMhSIbmozWxF`).")

The file 'data/processed/passport_index.faiss' exists.


In [50]:
import faiss

# Load the index to confirm it was created successfully
loaded_index = faiss.read_index("data/processed/passport_index.faiss")
print(f"Loaded FAISS index with {loaded_index.ntotal} vectors and dimension {loaded_index.d}.")

Loaded FAISS index with 33 vectors and dimension 768.


In [51]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "section": chunks[idx]["section"],
            "text": chunks[idx]["text"]
        })

    return results

In [52]:
results = retrieve("What are the fees to get a biometric passport?")

for result in results:
    print(result["score"])
    print(result["document"])
    print(result["section"])
    print(result["text"])
    print()

0.8796990513801575
Biometric Passport.txt
NB
To receive the passport immediately, the interested party can apply at the department of public relations. An additional fee of LBP 9,800,000 will be however requested.

0.8751458525657654
Biometric Passport.txt
Fees
Document requested	               Fees
Passport valid for 5 years	        6,000,000 L.L
Passport valid for 10 years	        10,000,000 L.L
Passport – first class - 5 years 	30,000,000 L.L
Passport – second class - 5 years 	20,000,000 L.L
Passport – first class - 10 years 	60,000,000 L.L
Passport – second class - 10 years 	40,000,000 L.L

0.8644660115242004
Biometric Passport.txt
Requested documents
The adequate application for passports format A4 (10 years) issued by the competent mayor according to the place of residence.
Lebanese ID card OR/AND an extract of civil status (whether the Lebanese citizen is applying for the 1st time for a biometric passport or not). Follow this link for more information: https://www.general-securi

In [65]:
!git config --global user.email "yarajnoun@gmail.com"
!git config --global user.name "yaranoun"

In [67]:
!git add data/processed/passport_index.faiss
!git commit -m "Update document FAISS"
!git pull --rebase origin main
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.
fatal: could not read Username for 'https://github.com': No such device or address
